# Sprint 4B — Full Controlled CUDA Training
Run all cells in order. Rerunning after interruption is safe because completed persistent runs are revalidated before being skipped.

## 1. Runtime and GPU inspection

In [ ]:
from pathlib import Path
import os, platform, subprocess, sys
BUNDLE_PATH = Path('/content/drive/MyDrive/synthdet/aquarium-sprint4b-v1.zip')
PERSISTENT_OUTPUT_DIRECTORY = Path('/content/drive/MyDrive/synthdet/sprint4b')
REGIME_SELECTION = 'all'
DEVICE = '0'
EXPECTED_REPOSITORY_REVISION = 'd6751dce7910a2f09b6b0d524a845a7ed6ede1fe'
WORK_DIRECTORY = Path('/content/synthdet-sprint4b')
print(platform.platform(), sys.version)
subprocess.run(['nvidia-smi'], check=True)


## 2. Google Drive mounting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSISTENT_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)


## 3. Training-bundle upload or Drive path selection

In [ ]:
assert BUNDLE_PATH.is_file(), f'Put the bundle at {BUNDLE_PATH} or change BUNDLE_PATH above'
SHA256_SIDECAR = BUNDLE_PATH.with_suffix(BUNDLE_PATH.suffix + '.sha256')
assert SHA256_SIDECAR.is_file(), f'Missing checksum sidecar: {SHA256_SIDECAR}'
print(BUNDLE_PATH, BUNDLE_PATH.stat().st_size)


## 4. Bundle SHA-256 validation

In [ ]:
import hashlib
expected_sha256 = SHA256_SIDECAR.read_text().split()[0]
digest = hashlib.sha256()
with BUNDLE_PATH.open('rb') as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b''): digest.update(block)
assert digest.hexdigest() == expected_sha256, 'Bundle SHA-256 mismatch'
print('Bundle SHA-256 verified:', expected_sha256)


## 5. Safe extraction

In [ ]:
import shutil, zipfile
if WORK_DIRECTORY.exists(): shutil.rmtree(WORK_DIRECTORY)
WORK_DIRECTORY.mkdir()
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    root = WORK_DIRECTORY.resolve()
    for member in archive.infolist():
        target = (WORK_DIRECTORY / member.filename).resolve()
        assert target == root or root in target.parents, f'Unsafe member: {member.filename}'
    archive.extractall(WORK_DIRECTORY)
os.chdir(WORK_DIRECTORY)
print('Extracted:', WORK_DIRECTORY)


## 6. Pinned dependency installation

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'configs/training/runtime-lock.txt'], check=True)


## 7. CUDA and PyTorch verification

In [ ]:
import torch, ultralytics
assert torch.cuda.is_available(), 'CUDA-enabled PyTorch is required'
tensor = torch.arange(64, device=f'cuda:{DEVICE}', dtype=torch.float32).reshape(8, 8)
product = tensor @ tensor.T
torch.cuda.synchronize(int(DEVICE))
assert product.is_cuda and torch.isfinite(product).all()
print(torch.__version__, torch.version.cuda, ultralytics.__version__, torch.cuda.get_device_name(int(DEVICE)))


## 8. Repository revision and frozen-identity validation

In [ ]:
inventory = __import__('json').loads(Path('training_bundle_inventory.json').read_text())
assert inventory['expected_repository_revision'] == EXPECTED_REPOSITORY_REVISION
subprocess.run([sys.executable, 'scripts/validate_training_bundle.py', '--extracted-root', '.', '--skip-experiment-validation'], check=True)


## 9. Experiment-view validation

In [ ]:
subprocess.run([sys.executable, 'scripts/materialize_training_bundle.py'], check=True)
subprocess.run([sys.executable, 'scripts/validate_experiments.py'], check=True)


## 10. GPU batch-profile preflight

In [ ]:
import urllib.request
weight_url = 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n.pt'
weight_path = Path('yolo11n.pt')
if not weight_path.is_file(): urllib.request.urlretrieve(weight_url, weight_path)
LOCAL_PROFILE = Path('artifacts/experiments/final_profile.json')
PERSISTENT_PROFILE = PERSISTENT_OUTPUT_DIRECTORY / 'final_profile.json'
if PERSISTENT_PROFILE.is_file():
    LOCAL_PROFILE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PERSISTENT_PROFILE, LOCAL_PROFILE)
    print('Reusing the already frozen persistent profile')
else:
    subprocess.run([sys.executable, 'scripts/gpu_preflight.py', '--device', DEVICE, '--output', str(LOCAL_PROFILE)], check=True)


## 11. Final profile freezing

In [ ]:
import json
profile = json.loads(LOCAL_PROFILE.read_text())
assert profile['status'] == 'frozen' and profile['batch'] in (16, 4) and profile['imgsz'] == 640
if PERSISTENT_PROFILE.exists():
    assert json.loads(PERSISTENT_PROFILE.read_text())['profile_identity'] == profile['profile_identity']
else:
    shutil.copy2(LOCAL_PROFILE, PERSISTENT_PROFILE)
print(profile['profile_name'], profile['profile_identity'])


## 12. Sequential five-regime training

In [ ]:
subprocess.run([sys.executable, 'scripts/colab_train.py', '--repository', str(WORK_DIRECTORY), '--expected-revision', EXPECTED_REPOSITORY_REVISION, '--regime', REGIME_SELECTION, '--device', DEVICE, '--persistent-output', str(PERSISTENT_OUTPUT_DIRECTORY), '--profile', str(PERSISTENT_PROFILE)], check=True)


## 13. Per-regime output validation

In [ ]:
state = json.loads((PERSISTENT_OUTPUT_DIRECTORY / 'completion_state.json').read_text())
required = ['synthetic_only', 'real_25', 'real_50', 'real_75', 'real_only'] if REGIME_SELECTION == 'all' else [REGIME_SELECTION]
assert all(state['regimes'][name]['status'] == 'completed' for name in required)
assert state['test_set_access_count'] == 0
print({name: state['regimes'][name]['best_pt_sha256'] for name in required})


## 14. Persistent Drive output copy

In [ ]:
assert all(Path(state['regimes'][name]['persistent_run_directory']).is_dir() for name in required)
print('Validated persistent directories before finalization')


## 15. Final training-completion audit

In [ ]:
assert REGIME_SELECTION == 'all', 'All five primary regimes are required for final audit'
subprocess.run([sys.executable, 'scripts/finalize_training_results.py', '--runs-root', str(PERSISTENT_OUTPUT_DIRECTORY / 'runs'), '--profile', str(PERSISTENT_PROFILE), '--output-directory', str(PERSISTENT_OUTPUT_DIRECTORY / 'return')], check=True)
completion = json.loads((PERSISTENT_OUTPUT_DIRECTORY / 'return/training_completion_manifest.json').read_text())
assert completion['status'] == 'completed' and completion['test_set_access_count'] == 0 and len(completion['runs']) == 5


## 16. Results archive creation

In [ ]:
RESULTS_ARCHIVE = PERSISTENT_OUTPUT_DIRECTORY / 'return/sprint4b_training_results.zip'
RESULTS_SHA256 = RESULTS_ARCHIVE.with_suffix(RESULTS_ARCHIVE.suffix + '.sha256')
RESULTS_INVENTORY = RESULTS_ARCHIVE.with_suffix(RESULTS_ARCHIVE.suffix + '.inventory.json')
assert RESULTS_ARCHIVE.is_file() and RESULTS_SHA256.is_file() and RESULTS_INVENTORY.is_file()
actual = hashlib.sha256(RESULTS_ARCHIVE.read_bytes()).hexdigest()
assert actual == RESULTS_SHA256.read_text().split()[0]
print('RETURN THESE FILES:', RESULTS_ARCHIVE, RESULTS_SHA256, RESULTS_INVENTORY, PERSISTENT_OUTPUT_DIRECTORY / 'return/training_completion_manifest.json', PERSISTENT_PROFILE)
